In [ ]:
from langchain_groq import ChatGroq
import os

In [ ]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key=os.environ["GROQ_API_KEY"],   
    model_name="openai/gpt-oss-120b"
)

# response = llm.invoke("The first person to land on moon was ...")
# print(response.content)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.amazon.jobs/en/jobs/10534303/software-development-manager-software-development")
page_data = loader.load().pop().page_content
# print(page_data)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

In [ ]:
type(json_res)

In [ ]:
# import pandas as pd

# df = pd.read_csv("my_portfolio.csv")
# df

In [ ]:
# import uuid
# import chromadb

# client = chromadb.PersistentClient('D:\LLM\coldEmail(LLM)')
# collection = client.get_or_create_collection(name="portfolio")

# if not collection.count():
#     for _, row in df.iterrows():
#         collection.add(documents=row["Techstack"],
#                        metadatas={"links": row["Links"]},
#                        ids=[str(uuid.uuid4())])

In [ ]:
job = json_res
job['skills']

In [ ]:
# links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
# links
from pypdf import PdfReader

reader = PdfReader("Pravesh_Resume(0).pdf")
resume_text = "\n".join(
    page.extract_text() or "" for page in reader.pages
)

print(resume_text) 

In [ ]:
# prompt_email = PromptTemplate.from_template(
#         """
#         ### JOB DESCRIPTION:
#         {job_description}
        
#         ### INSTRUCTION:
#         You are Pravesh Aggarwal, a Software developer  at Mobzway Technologies. Mobzway is a game development company specialize in Online Multi-player gaming products. 
#         pravesh have an hands-on experience with React.js, Node.js, Express.js, MongoDB, REST APIs, and JWT authentication
#         Your job is to write a cold email to the hr regarding the job mentioned above describing the capability of Pravesh.
#         email will be short so that it will be easily readable by the hr
#         Remember you are pravesh. 
#         also take help from the resume {reader} for writing the cold email. email will look like an proffesional 
#         Do not provide a preamble.
#         ### EMAIL (NO PREAMBLE):
        
#         """
#         )

# chain_email = prompt_email | llm
# res = chain_email.invoke({"job_description": str(job), "link_list": reader})
# # res = chain_email.invoke({"job_description": str(job)})
# print(res.content)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### CANDIDATE RESUME (the ONLY source of facts about the candidate):
    {resume}

    ### INSTRUCTION:
    You are the candidate described in the resume above. Write a cold email in the
    first person to the HR / hiring team for the job described above.

    SOURCE RULES (strict):
    - Use ONLY information that is explicitly written in the resume. Do not add,
      assume, or exaggerate any skill, tool, project, company, achievement, number,
      or year of experience.
    - Take the candidate's name, current role, employer, and contact details
      (email, phone, LinkedIn, GitHub) from the resume. Include only the ones
      that appear there.
    - Mention work experience (roles, companies, responsibilities, projects) only
      as the resume states it. State years of experience only if the resume's dates
      make it clear.
    - If the job asks for something the resume does not show, leave it out.
      Never claim it and never apologize for it.

    CONTENT RULES:
    - From the job description, pick the 3-4 requirements that matter most, then
      show the matching skills and experience from the resume as proof. Prefer
      concrete evidence (what was built, with which technologies) over generic
      claims like "hard-working" or "passionate".
    - Use the job title from the job description when referring to the role.

    FORMAT RULES:
    - Start with one line: "Subject: ..." (specific, mentions the role and the
      candidate's strongest relevant skill).
    - Then: greeting, a one-sentence introduction, a short paragraph (or 2-3 short
      lines) of matching experience, one sentence offering to share more or
      discuss, and a polite closing.
    - Sign off with the candidate's name and contact details from the resume.
    - Professional, confident, natural tone. No buzzwords, no emojis.
    - Maximum 150 words in the body.
    - No placeholders like [Your Name] or [Company]. If a detail is missing, omit it.
    - Output only the email. No preamble, no notes, no explanation.

    ### EMAIL:
    """
)

In [ ]:
import json
from langchain_core.output_parsers import StrOutputParser

chain_email = prompt_email | llm | StrOutputParser()

email = chain_email.invoke({
    "job_description": json.dumps(job, indent=2),
    "resume": resume_text,
})

print(email)